# Module 3: LLM Inference

> **Goal:** Understand how an LLM generates the final response after predicting the probability distribution for the next token.

---

# Inference Pipeline

```text
User Prompt
      ↓
Tokenizer
      ↓
Embeddings
      ↓
Transformer
      ↓
Probability Distribution
      ↓
Temperature
      ↓
Top-K
      ↓
Top-P
      ↓
Select Next Token
      ↓
Repeat Until Stop Condition
      ↓
Generated Response
```

---

# 13. Temperature ⭐⭐⭐⭐⭐

## What is Temperature?

Temperature controls **how random or deterministic** the model's response will be.

It **does not change the model's knowledge**.

It only changes **how the next token is selected**.

---

## Interview Definition

> **Temperature is a decoding parameter that controls the randomness of token selection during text generation.**

---

## Example

Probability Distribution

| Token | Probability |
|--------|------------:|
| Paris | 90% |
| London | 5% |
| Berlin | 3% |
| Rome | 2% |

### Temperature = 0

Always chooses

```
Paris
```

---

### Temperature = 1

Sometimes

```
Paris

or

London

or

Berlin
```

depending on probability.

---

## When to Use

| Temperature | Best For |
|-------------|----------|
| 0.0 | RAG |
| 0.2 | AI Agents |
| 0.3 | Code Generation |
| 0.7 | Chatbots |
| 1.0 | Creative Writing |

---

## Small Coding Example

```python
response = client.responses.create(
    model="gpt-4.1",
    input="Explain Python",
    temperature=0.2
)
```

---

## Interview Question

**Q. Does Temperature change model knowledge?**

**Answer:**

No.

It only changes the randomness during token selection.

---

# 14. Top-K Sampling ⭐⭐⭐⭐

## What is Top-K?

Top-K tells the model:

> **"Only consider the K most probable tokens."**

All remaining tokens are discarded.

---

## Interview Definition

> **Top-K sampling limits token selection to the K highest-probability tokens before choosing the next token.**

---

## Example

Probability Distribution

| Token | Probability |
|--------|------------:|
| Paris | 40% |
| London | 25% |
| Berlin | 15% |
| Rome | 10% |
| Tokyo | 5% |
| Delhi | 5% |

---

### Top-K = 3

Allowed

```
Paris

London

Berlin
```

Ignored

```
Rome

Tokyo

Delhi
```

The next token will be selected only from the top three.

---

## Why Use Top-K?

Without Top-K

```
Thousands of possible tokens
```

With Top-K

```
Only Top K candidates
```

More controlled generation.

---

## Small Coding Example

```python
response = client.responses.create(
    model="gpt-4.1",
    input="Tell me a joke.",
    top_k=40
)
```

*(Some APIs expose `top_k`; others do not.)*

---

## Interview Question

**Q. What happens if Top-K = 1?**

**Answer:**

Only the highest-probability token is selected.

This behaves similarly to greedy decoding.

---

# 15. Top-P Sampling (Nucleus Sampling) ⭐⭐⭐⭐⭐

## What is Top-P?

Instead of selecting a fixed number of tokens,

Top-P selects the **smallest group of tokens whose cumulative probability reaches P**.

---

## Interview Definition

> **Top-P sampling selects the smallest set of tokens whose cumulative probability is greater than or equal to P, then samples from that set.**

---

## Example

Probability Distribution

| Token | Probability | Cumulative |
|--------|------------:|-----------:|
| Paris | 50% | 50% |
| London | 25% | 75% |
| Berlin | 15% | 90% |
| Rome | 5% | 95% |
| Tokyo | 5% | 100% |

---

### Top-P = 0.90

Allowed

```
Paris

London

Berlin
```

Ignored

```
Rome

Tokyo
```

Unlike Top-K,

the number of selected tokens changes dynamically.

---

## Why Top-P is Better Than Top-K?

Suppose

Prompt A

```
Only two tokens are highly probable.
```

Prompt B

```
Ten tokens are highly probable.
```

Top-K always selects a fixed number.

Top-P adapts automatically.

---

## Small Coding Example

```python
response = client.responses.create(
    model="gpt-4.1",
    input="Write a poem.",
    top_p=0.9
)
```

---

## Interview Question

**Q. Difference between Top-K and Top-P?**

**Answer:**

Top-K selects a fixed number of tokens.

Top-P selects a dynamic number based on cumulative probability.

---

# 16. Max Tokens ⭐⭐⭐

## What is Max Tokens?

Max Tokens defines

> **The maximum number of tokens the model is allowed to generate.**

---

## Interview Definition

> **Max Tokens limits the maximum length of the generated response.**

---

## Example

Prompt

```
Explain Artificial Intelligence.
```

### max_tokens = 20

```
Artificial Intelligence is the simulation of human intelligence...
```

(Response stops quickly.)

---

### max_tokens = 300

Detailed explanation.

---

## Why Use It?

- Control API cost
- Reduce latency
- Prevent excessively long responses

---

## Small Coding Example

```python
response = client.responses.create(
    model="gpt-4.1",
    input="Explain LangGraph",
    max_output_tokens=150
)
```

---

## Interview Question

**Q. Does Max Tokens limit the input?**

**Answer:**

No.

It limits only the generated output.

---

# 17. Stop Sequences ⭐⭐⭐

## What are Stop Sequences?

Stop Sequences tell the model:

> **"Stop generating text when this sequence appears."**

---

## Interview Definition

> **A Stop Sequence is a predefined token or string that instructs the model to terminate text generation when encountered.**

---

## Example

Prompt

```
Generate SQL.
```

Stop Sequence

```text
;
```

Generated

```sql
SELECT * FROM Employees;
```

Generation stops after

```
;
```

---

Another example

Stop Sequence

```
END
```

Generated

```text
Step 1

Step 2

END
```

The model stops immediately.

---

## Small Coding Example

```python
response = client.responses.create(
    model="gpt-4.1",
    input="Generate SQL query",
    stop=[";"]
)
```

---

## Why Use Stop Sequences?

- Structured Output
- Chatbots
- SQL Generation
- Code Generation
- API Responses

---

## Interview Question

**Q. Why are Stop Sequences useful?**

**Answer:**

They prevent unnecessary generation and help produce structured, predictable outputs.

---

# Comparison Table ⭐⭐⭐⭐⭐

| Parameter | Purpose | Example |
|-----------|---------|---------|
| Temperature | Controls randomness | Creative vs deterministic output |
| Top-K | Selects from top K tokens | Top 40 tokens |
| Top-P | Selects tokens until cumulative probability reaches P | Top 90% probability |
| Max Tokens | Limits response length | 200 output tokens |
| Stop Sequences | Stops generation at a specified token/string | Stop at `;` or `END` |

---

# Small Coding Example (All Together)

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-4.1",
    input="Explain LangGraph.",
    temperature=0.2,
    top_p=0.9,
    max_output_tokens=200,
    stop=["END"]
)

print(response.output_text)
```

> **Note:** Some LLM providers expose `top_k`, while others (such as OpenAI's current Responses API) expose `temperature`, `top_p`, and `max_output_tokens` but not `top_k`.

---

# Common Interview Questions

### Q1. Difference between Temperature and Top-P?

**Answer**

- Temperature changes the randomness of the probability distribution.
- Top-P chooses tokens from the smallest cumulative probability set.

---

### Q2. Difference between Top-K and Top-P?

**Answer**

Top-K selects a fixed number of tokens.

Top-P selects a variable number of tokens based on cumulative probability.

---

### Q3. Which parameter controls response length?

**Answer**

Max Tokens.

---

### Q4. Which parameter makes the response more creative?

**Answer**

Higher Temperature.

---

### Q5. Which parameter is commonly kept low in RAG?

**Answer**

Temperature (around 0–0.2).

---

### Q6. What are Stop Sequences used for?

**Answer**

To terminate generation when a predefined token or string appears.

---

# Quick Revision

| Parameter | One-Line Summary |
|-----------|------------------|
| Temperature | Controls randomness |
| Top-K | Keep only top K candidate tokens |
| Top-P | Keep tokens until cumulative probability reaches P |
| Max Tokens | Maximum response length |
| Stop Sequences | Stops generation when a specified sequence appears |

---

# Interview Cheat Sheet

```text
Probability Distribution
          │
          ▼
   Temperature
          │
          ▼
      Top-K
          │
          ▼
      Top-P
          │
          ▼
  Select Next Token
          │
          ▼
Generate Response
          │
          ▼
Stop Sequence OR Max Tokens
```

---

# 30-Second Interview Answer

> **After an LLM predicts the probability distribution for the next token, inference parameters determine how the final response is generated. Temperature controls randomness, Top-K limits selection to the top K tokens, Top-P selects from the smallest set of tokens whose cumulative probability reaches a threshold, Max Tokens limits the response length, and Stop Sequences terminate generation when a specified sequence is encountered. These parameters allow developers to balance accuracy, creativity, cost, and output structure.**